In [1]:
# ============================================================
# Amazon UK Price Prediction — Modeling Phase (Step 1)
# Plan: group split -> baseline -> linear -> LightGBM
# Har step pe RMSE compare. Target = log_price.
# ============================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_squared_error

df = pd.read_parquet("amz_clean.parquet")
print(df.shape)

# ------------------------------------------------------------
# STEP 1 — GROUP SPLIT BY ASIN (leakage prevention)
# Same product train+test dono mein nahi jana chahiye
# ------------------------------------------------------------
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["asin"]))

train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

# verify: koi asin overlap to nahi?
overlap = set(train["asin"]) & set(test["asin"])
print(f"Train: {train.shape}, Test: {test.shape}")
print(f"ASIN overlap: {len(overlap)}")   # MUST be 0

y_train = train["log_price"]
y_test = test["log_price"]

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

results = {}

# ------------------------------------------------------------
# STEP 2 — BASELINE (median predictor)
# Har model ko isse behtar hona HAI, warna model bekaar hai
# ------------------------------------------------------------
median_pred = np.full(len(y_test), y_train.median())
results["Baseline (median)"] = rmse(y_test, median_pred)
print(f"Baseline RMSE (log scale): {results['Baseline (median)']:.4f}")




(2441960, 13)
Train: (1953587, 13), Test: (488373, 13)
ASIN overlap: 0
Baseline RMSE (log scale): 1.2827


In [2]:
# ------------------------------------------------------------
# STEP 3 — FEATURES
# Round 1: simple numeric + flags. Title/category encoding baad mein.
# ------------------------------------------------------------
features = ["stars", "log_reviews", "has_rating", "had_sales",
            "isBestSeller", "boughtInLastMonth"]

X_train = train[features].astype(float)
X_test = test[features].astype(float)

In [3]:
# ------------------------------------------------------------
# STEP 4 — LINEAR REGRESSION
# ------------------------------------------------------------
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)
results["Linear Regression"] = rmse(y_test, lr.predict(X_test))
print(f"Linear RMSE: {results['Linear Regression']:.4f}")


Linear RMSE: 1.2217


In [6]:
import pickle

with open("linear_regression.pkl", "wb") as f:
    pickle.dump(lr, f)

In [4]:
# ------------------------------------------------------------
# STEP 5 — LIGHTGBM
# pip install lightgbm --break-system-packages (agar nahi hai)
# ------------------------------------------------------------
import lightgbm as lgb

# LightGBM category natively handle karta hai — bina one-hot ke!
features_lgb = features + ["category"]
X_train_lgb = train[features_lgb].copy()
X_test_lgb = test[features_lgb].copy()

model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    random_state=42,
    n_jobs=-1,
)
model.fit(
    X_train_lgb, y_train,
    eval_set=[(X_test_lgb, y_test)],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
)
results["LightGBM"] = rmse(y_test, model.predict(X_test_lgb))
print(f"LightGBM RMSE: {results['LightGBM']:.4f}")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.044652 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 578
[LightGBM] [Info] Number of data points in the train set: 1953587, number of used features: 7
[LightGBM] [Info] Start training from score 3.272734
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l2: 1.14317
[200]	valid_0's l2: 1.13997
[300]	valid_0's l2: 1.13883
[400]	valid_0's l2: 1.13814
[500]	valid_0's l2: 1.13758
Did not meet early stopping. Best iteration is:
[500]	valid_0's l2: 1.13758
LightGBM RMSE: 1.0666


In [7]:
with open("lightgbm.pkl", "wb") as f:
    pickle.dump(model, f)

In [5]:
# ------------------------------------------------------------
# STEP 6 — COMPARE + INTERPRET
# ------------------------------------------------------------
print("\n========== RESULTS (RMSE on log_price) ==========")
for name, score in results.items():
    print(f"{name:22} : {score:.4f}")

# RMSE ko samajhne layak banao: log-scale error -> "% error" feel
best = min(results.values())
print(f"\nBest log-RMSE {best:.3f} ~ typical error factor of "
      f"{np.exp(best):.2f}x on actual price")
# e.g. 0.69 -> ~2x: agar asli price £20 hai to model typically £10-£40 bolta hai

# Feature importance — konsa feature kaam kar raha hai
imp = pd.Series(model.feature_importances_, index=features_lgb)
print("\nFeature importance:")
print(imp.sort_values(ascending=False))


========== RESULTS (RMSE on log_price) ==========
Baseline (median)      : 1.2827
Linear Regression      : 1.2217
LightGBM               : 1.0666

Best log-RMSE 1.067 ~ typical error factor of 2.91x on actual price

Feature importance:
log_reviews          11822
category             10866
stars                 6382
boughtInLastMonth     1201
had_sales              538
isBestSeller           170
has_rating              21
dtype: int32


In [9]:
import pickle

with open("linear_regression.pkl", "rb") as f:
    lr = pickle.load(f)

with open("lightgbm.pkl", "rb") as f:
    model = pickle.load(f)


In [11]:
import os

os.makedirs("models", exist_ok=True)
pickle.dump(lr, open("models/linear_regression.pkl", "wb"))
pickle.dump(model, open("models/lightgbm.pkl", "wb"))
pickle.dump(features_lgb, open("models/features.pkl", "wb"))